# A reproducible GeoAI framework for explainable spatiotemporal analysis of national missing-person registers (Peru, 2019–2025)

**Reproducibility notebook: data, code and results of the article submitted to *Information Sciences*.**

Moisés Evangelista Gamarra ([0009-0002-5382-1390](https://orcid.org/0009-0002-5382-1390)) · Edwar Abrahan Alama Carreño ([0009-0006-8029-4302](https://orcid.org/0009-0006-8029-4302)) · Pamela Estefani Figueroa Rosas ([0009-0007-6100-9382](https://orcid.org/0009-0007-6100-9382))
Servicio Nacional de Adiestramiento en Trabajo Industrial (SENATI), Lima, Peru

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USER/GeoAI-MissingPersons-Peru/blob/main/notebooks/GeoAI_MissingPersons_Peru.ipynb)
[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.XXXXXXX.svg)](https://doi.org/10.5281/zenodo.XXXXXXX)

| | |
|---|---|
| Package / analysis plan | `reniped` 3.3.2 · plan v3.3.2 |
| Plan SHA-256 | `3b6e0f676646baed271a107db441579d58230bbcec607f85a5f45d7e7e1f2958` |
| RENIPED data SHA-256 | `c41ebcced14fdf86bbf17db9f54bb381d73c0464a402f22ac5548a7899ef0e32` |
| Random seed | 42 |

## Quick Start

1. Open this notebook in Google Colab (badge above).
2. Select **Runtime ▸ Run all**.

No GPU and no local setup are needed. A full run takes about **30 minutes** on a free Colab CPU (about 25 of them in Section 11). Every number of the article is regenerated into `results/`. The last section compares the new tables with the reference tables of the run reported in the article.

## Contents (manuscript sections in brackets)

0. Environment setup [4.7]
1. Data sourcing and integrity [3]
2. Automated tests [4.7]
3. Analysis plan freeze [4.7]
4. Territory-month panel [3]
5. Rates and priority territories [4.1, 5.1; Figs. 2–3]
6. Features and feature spaces [4.2; Table 1]
7. Choice of the number of states [4.3, 5.2; Fig. 4a–c]
8. Multi-algorithm clustering and state profiles [4.3, 5.2; Fig. 4d]
9. Multilevel validation I: territorial-identity audit [4.6, 5.5; Table 3A]
10. Temporal dynamics [4.5, 5.4]
11. Spatial structure [4.4, 5.3; Table 2]
12. Multilevel validation II: specification multiverse [4.6, 5.5; Fig. 5, Table 3B]
13. Article figures [Figs. 1–5]
14. Export and reproduction check

The identity audit (Section 9) runs before the spatial analysis because the spatial stage also tests the share of months in the high state, which the audit stage computes.

Console messages, table names and column names are in Spanish, as in the frozen analysis plan. The repository `README.md` has a glossary.

## 0. Environment setup

When this notebook runs outside the repository (for example, in a fresh Colab runtime), it clones the repository at the release cited in the article. Run locally from `notebooks/`, it uses the repository it sits in.

In [ ]:
# 0a. Locate the repository, or clone it on Colab
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/YOUR-GITHUB-USER/GeoAI-MissingPersons-Peru.git"
REPO_REF = "v3.3.2"          # release cited in the article; falls back to the default branch
IN_COLAB = "google.colab" in sys.modules


def find_repo_root(start):
    """Return the first parent directory that contains the reniped package."""
    for p in (start, *start.parents):
        if (p / "src" / "reniped").is_dir():
            return p
    return None


ROOT = find_repo_root(Path.cwd())
if ROOT is None:
    ROOT = Path("/content/GeoAI-MissingPersons-Peru") if IN_COLAB else Path.cwd() / "GeoAI-MissingPersons-Peru"
    if not ROOT.exists():
        r = subprocess.run(["git", "clone", "--quiet", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(ROOT)])
        if r.returncode != 0:     # release tag not found: use the default branch
            subprocess.run(["git", "clone", "--quiet", "--depth", "1", REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Repository root:", ROOT)

The pinned versions of the reference run are in `requirements.txt`. Colab already ships most of them. The cell below installs only what is missing, with the exact reference version, and reports any difference. HDBSCAN comes from scikit-learn (≥ 1.3), not from the standalone `hdbscan` package.

To force the exact reference versions, set `EXACT_PINS = True`, run the cell, restart the session (**Runtime ▸ Restart session**) and run everything again.

In [ ]:
# 0b. Install missing libraries with the reference versions and compare the environment
import importlib
import importlib.metadata as md

EXACT_PINS = False

PINS = dict(line.strip().split("==") for line in (ROOT / "requirements.txt").read_text().splitlines()
            if "==" in line)


def installed(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None


to_install = list(PINS) if EXACT_PINS else [p for p in PINS if installed(p) is None]
if to_install:
    print("Installing:", ", ".join(f"{p}=={PINS[p]}" for p in to_install))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[f"{p}=={PINS[p]}" for p in to_install]],
                   check=True)
    importlib.invalidate_caches()
    if EXACT_PINS:
        print("Exact versions installed. Restart the session and run all cells again.")

print(f"\n{'package':<14}{'reference':>12}{'installed':>12}")
mismatch = []
for pkg, ref in PINS.items():
    cur = installed(pkg)
    flag = "" if cur == ref else "  <- differs"
    if flag:
        mismatch.append(pkg)
    print(f"{pkg:<14}{ref:>12}{str(cur):>12}{flag}")
print(f"\nPython {sys.version.split()[0]} (reference run: 3.13.15)")
if mismatch:
    print("Note: versions differ from the reference run for", ", ".join(mismatch) + ".",
          "Results are expected to match; the check in Section 14 reports any difference.")

In [ ]:
# 0c. Imports, global seed and a copy of the reference tables for the final check
import json
import shutil
import tempfile
import time

import numpy as np
import pandas as pd

from reniped import __version__
from reniped import pipeline as P
from reniped.config import PLAN
from reniped.io_utils import sha256_file
from reniped.results import freeze_environment, write_manifest

SEED = PLAN["semilla"]
assert SEED == 42
np.random.seed(SEED)              # global seed; every stage also receives SEED explicitly
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

# The repository ships the tables of the run reported in the article. They are copied before
# the pipeline overwrites them, so that Section 14 can compare old and new values.
REF_TABLES = Path(tempfile.gettempdir()) / "reference_tables_v3.3.2"
if (ROOT / "results" / "tables").is_dir() and not REF_TABLES.exists():
    shutil.copytree(ROOT / "results" / "tables", REF_TABLES)
T0 = time.time()
print(f"reniped {__version__} | analysis plan {PLAN['version_plan']} | seed {SEED}")
print("Reference tables:", len(list(REF_TABLES.glob('*.csv'))) if REF_TABLES.exists() else 0)

## 1. Data sourcing and integrity (Section 3)

The three input files are included in the repository exactly as downloaded from the official sources, because the open-data portal can update the RENIPED resource. Each file must match, byte by byte, the SHA-256 reported in the article and stored in the frozen plan.

| File | Source |
|---|---|
| `data/raw/DATASET_Personas_Desaparecidas_2019-01_2025-12.csv` | MININTER, [Plataforma Nacional de Datos Abiertos](https://www.datosabiertos.gob.pe/dataset/personas-desaparecidas/resource/f49ae77f-8822-45ea-aa16-911d677e303d) (ODC-By) |
| `data/external/INEI_poblacion_proyectada_2018-2026_cuadro01.xlsx` | [INEI, population projections 2018–2026](https://www.gob.pe/institucion/inei/informes-publicaciones/6894980-peru-poblacion-total-proyectada-al-30-de-junio-de-cada-ano-segun-departamento-provincia-y-distrito-2018-2026), Table No. 01 |
| `data/external/limites_provinciales_peru_simplificados.geojson` | [juaneladio/peru-geojson](https://github.com/juaneladio/peru-geojson) (MPL-2.0) |

In [ ]:
# 1. Verify the SHA-256 of every input file against the frozen plan
paths = P.data_paths(ROOT)
meta = {"csv": PLAN["datos"]["csv_reniped"], "xlsx": PLAN["datos"]["xlsx_inei"],
        "geo": PLAN["datos"]["geometria"]}
SOURCES = {
    "csv": "https://www.datosabiertos.gob.pe/dataset/personas-desaparecidas/resource/f49ae77f-8822-45ea-aa16-911d677e303d",
    "xlsx": "https://www.gob.pe/institucion/inei/informes-publicaciones/6894980",
    "geo": PLAN["datos"]["geometria"]["url"],
}
for key, path in paths.items():
    if not path.exists():
        if key == "geo":           # the only file with a stable direct URL
            import urllib.request
            path.parent.mkdir(parents=True, exist_ok=True)
            urllib.request.urlretrieve(SOURCES["geo"], path)
        else:
            raise FileNotFoundError(f"{path.relative_to(ROOT)} is missing; it is part of the repository "
                                    f"(original source: {SOURCES[key]}).")
    digest = sha256_file(path)
    print(f"{path.relative_to(ROOT)}\n   SHA-256 {digest}  {'OK' if digest == meta[key]['sha256'] else 'MISMATCH'}")
    assert digest == meta[key]["sha256"], f"{path.name}: SHA-256 differs from the value in the article"
print("\nAll input files match the article byte by byte.")

## 2. Automated tests

These ten tests use synthetic data to check the critical components: the exact conditional p-values (including the single-neighbour case), the priority classification, the handling of missing months, semantic labels and strict JSON output.

In [ ]:
# 2. Run the automated tests (< 1 min)
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", "tests"], capture_output=True, text=True)
if "No module named pytest" in r.stderr:
    r = subprocess.run([sys.executable, "tests/test_core.py"], capture_output=True, text=True)
print(r.stdout[-3000:] or r.stderr[-3000:])
assert r.returncode == 0, "Some tests failed"

## 3. Analysis plan freeze (Section 4.7)

All analytical decisions are fixed in `PLAN` (`src/reniped/config.py`) before the run: data, rates, folds, features, K rule, algorithms, spatial weights, permutations and multiverse. The plan is written to `config/analysis_plan.json`, and its SHA-256 is checked against the lock file. The run stops if the plan changed without a version bump.

In [ ]:
# 3. Freeze the plan and open the results store
ctx = P.stage_init(ROOT)
assert ctx["res"].data["plan"]["sha256"].startswith("3b6e0f67"), "The analysis plan differs from the article"

## 4. Territory-month panel (Section 3)

This stage reads the 77,331 rows and 142,374 reports of the register and the INEI denominators. It builds the panel of 26 territories × 84 months (2019–2025): Lima is split into Lima Metropolitana and Región Lima, and Callao is kept separate. Months without source rows are coded as missing, never as zero, and the stage checks that they coincide with the documented coverage gaps (Section 3.3).

Output: `T01_tasas_departamentales_2019_2025.csv`.

In [ ]:
ctx = P.stage_data(ctx)

## 5. Rates and priority territories (Sections 4.1 and 5.1; Figs. 2–3)

Each territory's rate is compared with the rate of the **rest of the country**, not with the national total, which would pull large territories towards 1. The 95% intervals come from a 12-month moving-block bootstrap (9,999 replicates) that draws the same months for all territories. The Benjamini–Hochberg FDR control is applied across the 26 territories. Two sensitivity analyses are run: 2019–2024 only (2025 is preliminary) and total exposure.

Outputs: `T14_territorios_prioritarios*.csv` (Fig. 2) and `T14b_razon_tasas_anual.csv` (Fig. 3).

In [ ]:
ctx = P.stage_priority(ctx)

## 6. Features and feature spaces (Section 4.2; Table 1)

The features are computed per temporal fold, using only information available in the training years, so that nothing leaks from the future:

- **main fold:** train 2019–2023, evaluate 2024;
- **secondary fold:** train 2019–2024, evaluate 2025.

The spaces are E1 (relative: anomaly and 12-month change), E2 (mixed: E1 plus the prior 12-month level; the main space) and E3 (level). Rank and condition number are reported for each space.

In [ ]:
ctx = P.stage_features(ctx)
pd.DataFrame([(k, ", ".join(v)) for k, v in PLAN["features"]["espacios"].items()],
             columns=["feature space", "features (Table 1)"])

## 7. Choice of the number of states (Sections 4.3 and 5.2; Fig. 4a–c)

K is chosen with the gap statistic, starting from K = 1 to test whether there is any cluster structure at all. Bootstrap stability is measured in three ways: by whole territories, by blocks of 12 consecutive months and by rows. Resampling rows ignores the panel dependence and overstates stability. The pre-specified rule defines as candidates the K whose unit and temporal-block stability reach ARI ≥ 0.80. It then takes Tibshirani's gap K if it is a candidate, otherwise the candidate with the highest silhouette. If no K reaches the threshold, it takes the K with the highest minimum stability, which is what happens with these data (K = 4, while the gap statistic peaks at K = 1).

Outputs: `T02_seleccion_K.csv`, `T02b_gap_desde_K1.csv`.

In [ ]:
ctx = P.stage_k(ctx)

## 8. Multi-algorithm clustering and state profiles (Sections 4.3 and 5.2; Fig. 4d)

K-means, Ward and a Gaussian mixture are fitted at the chosen K, and HDBSCAN is fitted as a density-based check. States are relabelled semantically (0 = lowest anomaly), so their numbers are comparable across algorithms, folds and years.

Outputs: `T03_multialgoritmo.csv`, `T04*_...csv` (profiles, mean ± SD and standardized distributions).

In [ ]:
ctx = P.stage_algorithms(ctx)

## 9. Multilevel validation I: territorial-identity audit (Sections 4.6 and 5.5; Table 3, Panel A)

Do the clusters capture states shared across territories, or do they merely recover which territory each month belongs to? The audit measures NMI(cluster, territory), the fraction of the cluster entropy explained by the territory, and territory purity. The observed NMI is compared with a null that permutes territory identities within each year (999 permutations), which preserves the within-year sequence of every series.

Output: `T05_auditoria_proxy.csv`.

In [ ]:
ctx = P.stage_audit(ctx)

## 10. Temporal dynamics (Sections 4.5 and 5.4)

The model fitted on the training years assigns states in the evaluation year (2024; 2025 as secondary). The transferred labels are compared with a refit on that year (ARI), and the state shares of the two periods are compared with the Jensen–Shannon distance.

Outputs: `T06_proporciones_anuales_*.csv`.

In [ ]:
ctx = P.stage_temporal(ctx)

## 11. Spatial structure (Sections 4.4 and 5.3; Table 2)

The spatial stage uses two weight matrices: queen contiguity and the four nearest neighbours by geodesic distance. It computes:

- global Moran's I (9,999 permutations; two-sided p), plus an empirical-Bayes version for rates;
- local Gi\* and LISA with **exact** conditional-randomization p-values;
- Benjamini–Hochberg FDR control, with Benjamini–Yekutieli as a sensitivity analysis;
- a leave-one-out influence check.

Outputs: `T07`–`T11`.

In [ ]:
ctx = P.stage_spatial(ctx)

## 12. Multilevel validation II: specification multiverse (Sections 4.6 and 5.5; Fig. 5, Table 3, Panel B)

- **M1 (clustering, 108 specifications):** unit of analysis × winsorization × feature space × scaler × rule for choosing K.
- **M2 (spatial):** counts or rates × exposure × period × weights × p-value type × local correction.

The specification of the initial analysis is included as a reproduction anchor. For each claim, the stage reports the share of specifications that support it, and the partial η² of each decision.

⏱ This is the longest stage: about 25 minutes on a free Colab CPU.

Outputs: `T12*` (M1) and `T13*` (M2).

In [ ]:
ctx = P.stage_multiverse(ctx)

## 13. Article figures (Figs. 1–5)

The figures are drawn at their final printed size (190 mm wide, text ≥ 7 pt), only from the saved tables. They are saved in `results/figures_article/` in English (`_en`, submission) and Spanish (`_es`), as PNG (preview, 300 dpi) and TIFF (submission; 600 dpi for the map, 1000 dpi for line art).

In [ ]:
ctx = P.stage_article_figures(ctx)
from IPython.display import Image, display
for name in ("Fig1_framework", "Fig2_priority", "Fig3_persistence", "Fig4_states", "Fig5_validation"):
    print(name)
    display(Image(filename=str(ROOT / "results" / "figures_article" / f"{name}_en.png"), width=900))

## 14. Export and reproduction check

This section:

1. saves every number to `results/FINAL_RESULTS.json`, the software and hardware of this run to `results/environment.json`, and the SHA-256 of every file to `MANIFEST.sha256`;
2. compares the 35 regenerated tables with the reference tables of the run reported in the article.

Only the HDBSCAN internal indices in `T03_multialgoritmo.csv`, which the article does not report, may differ across machines.

In [ ]:
# 14a. Save results, environment and manifest
res = ctx["res"]
res.set("entorno", freeze_environment(ROOT / "results"))
print("Saved:", res.save().relative_to(ROOT))
print("Files in MANIFEST.sha256:", write_manifest(ROOT))
print(f"Total run time: {(time.time() - T0) / 60:.1f} min")

In [ ]:
# 14b. Compare every regenerated table with the reference run
def compare_tables(ref_dir, new_dir, rtol=1e-6, atol=1e-9):
    rows = []
    for f in sorted(ref_dir.glob("*.csv")):
        g = new_dir / f.name
        if not g.exists():
            rows.append({"table": f.stem, "status": "MISSING", "max_abs_diff": None, "rows_differing": None})
            continue
        a, b = pd.read_csv(f), pd.read_csv(g)
        if a.shape != b.shape or list(a.columns) != list(b.columns):
            rows.append({"table": f.stem, "status": "DIFFERENT SHAPE", "max_abs_diff": None, "rows_differing": None})
            continue
        num = [c for c in a.columns if pd.api.types.is_numeric_dtype(a[c]) and pd.api.types.is_numeric_dtype(b[c])]
        txt = [c for c in a.columns if c not in num]
        x, y = a[num].to_numpy(float), b[num].to_numpy(float)
        close = np.isclose(x, y, rtol=rtol, atol=atol, equal_nan=True)
        same_txt = (a[txt].fillna("").astype(str).values == b[txt].fillna("").astype(str).values)
        bad = ~(close.all(axis=1) & same_txt.all(axis=1))
        diff = np.nanmax(np.abs(x - y)) if len(num) and np.isfinite(x - y).any() else 0.0
        rows.append({"table": f.stem, "status": "identical" if not bad.any() else "differs",
                     "max_abs_diff": float(diff), "rows_differing": int(bad.sum())})
    return pd.DataFrame(rows)


if REF_TABLES.exists():
    check = compare_tables(REF_TABLES, ROOT / "results" / "tables")
    print(check.status.value_counts().to_string(), "\n")
    print(check[check.status != "identical"].to_string(index=False) or "All tables identical to the reference run.")
    unexpected = check[(check.status != "identical") & (check.table != "T03_multialgoritmo")]
    if len(unexpected):
        print("\nWARNING: differences outside T03 (see above). Check the library versions of Section 0.")
    else:
        print("\nReproduction check passed: every table reported in the article matches the reference run.")
else:
    print("No reference tables found (results/tables was not in the repository copy).")

In [ ]:
# 14c. Headline numbers of the article, read from the regenerated results
d = res.data
prio = pd.read_csv(ROOT / "results/tables/T14_territorios_prioritarios.csv")
moran = pd.read_csv(ROOT / "results/tables/T09_moran_global.csv")
audit = pd.read_csv(ROOT / "results/tables/T05_auditoria_proxy.csv")
m = moran[(moran.periodo == "2019_2025") & (moran.variable == "TASA_10K_EXPOSICION_OBSERVADA")]
e2 = audit[(audit.historia == "min6") & (audit.espacio == "E2_relativo_mas_nivel")].iloc[0]
print("Priority classes (Fig. 2):", prio.CLASE.value_counts().to_dict())
print("Highest rate ratios:", ", ".join(f"{r.DPTO_KEY} {r.RR_VS_RESTO:.2f}" for r in prio.nlargest(5, "RR_VS_RESTO").itertuples()))
print("Chosen K (Fig. 4):", d["seleccion_k"]["K_elegido"], "|", d["seleccion_k"]["motivo"])
print("Global Moran's I, 2019-2025 rate:",
      "; ".join(f"{r.pesos} I = {r.I:.3f} (p = {r.p_bilateral:.3f})" for r in m.itertuples()))
print(f"Identity audit, space E2: NMI = {e2.NMI_cluster_unidad:.3f}, p = {e2.p_perm_bloques_anuales:.3f}")
print("Multiverse:", d["multiverso"]["M1"]["especificaciones"], "clustering and",
      d["multiverso"]["M2"]["especificaciones"], "spatial specifications")

In [ ]:
# 14d. Optional: download the results as a ZIP file (Colab)
if IN_COLAB:
    archive = shutil.make_archive("/content/GeoAI_MissingPersons_Peru_results", "zip", ROOT,
                                  base_dir="results")
    from google.colab import files
    files.download(archive)